# Document Preparation - Tree Pipeline

Demonstrates the semantic tree path:

```
PdfParser / DocxParser (mode="tree")
        ↓  Document.root  (TreeNode hierarchy)
RecursiveChunker  (tree-walk path)
        ↓  List[Chunk]
```

Key properties of the tree path:
- **Tables and images are always atomic** — never split across chunks.
- **Headings bind to their paragraphs** — a section and its body are kept together when they fit within `chunk_size`.
- **Leaf fallback** — oversized paragraphs are split last, after all structural boundaries are exhausted.

## Imports

In [ ]:
# Standard Library
import pathlib

# Third Party Library

# Private Library
from cleave.chunker.recursive import RecursiveChunker
from cleave.parsers.office.docx import DocxParser
from cleave.parsers.office.pdf import PdfParser
from cleave.schemas import ChunkParams, ContentType

## Fixtures

In [ ]:
FIXTURES  = pathlib.Path.cwd().parent.parent.parent / "tests" / "fixtures"
PDF_PATH  = str(FIXTURES / "sample.pdf")
DOCX_PATH = str(FIXTURES / "sample.docx")
print("PDF :", PDF_PATH)
print("DOCX:", DOCX_PATH)

## Parse (tree mode)

In [ ]:
pdf_doc  = PdfParser(PDF_PATH,   mode="tree").parse()
docx_doc = DocxParser(DOCX_PATH, mode="tree").parse()

def count_nodes(node):
    return 1 + sum(count_nodes(c) for c in node.children)

print(f"PDF  — tree nodes: {count_nodes(pdf_doc.root)}")
print(f"DOCX — tree nodes: {count_nodes(docx_doc.root)}")

## Inspect the DOCX tree

In [ ]:
def print_tree(node, indent=0):
    role  = node.metadata.get("role", "")
    level = node.metadata.get("level", "")
    label = f"[{node.content_type.value}]"
    if role:  label += f" role={role}"
    if level: label += f" level={level}"
    preview = node.content[:70].replace("\n", " ") if node.content else ""
    print("  " * indent + f"{label}  \"{preview}\"")
    for child in node.children:
        print_tree(child, indent + 1)

print_tree(docx_doc.root)

## RecursiveChunker on DOCX tree

In [ ]:
CHUNK_SIZE    = 300
CHUNK_OVERLAP = 30

chunker = RecursiveChunker(ChunkParams(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP))
chunks  = chunker.chunk(docx_doc)

print(f"chunk_size={CHUNK_SIZE}  overlap={CHUNK_OVERLAP}  total={len(chunks)}\n")
for c in chunks:
    print(f"[{c.index}] type={c.content_type.value:5}  tokens={c.token_count:>3}  │  {c.text[:80]!r}")

### Atomicity check
Tables and images must always appear as exactly one chunk, regardless of size.

In [ ]:
table_chunks = [c for c in chunks if c.content_type == ContentType.table]
image_chunks = [c for c in chunks if c.content_type == ContentType.image]

print(f"Table chunks : {len(table_chunks)}")
print(f"Image chunks : {len(image_chunks)}")

if table_chunks:
    print("\nTable chunk content:")
    print(table_chunks[0].text)

## RecursiveChunker on PDF tree

In [ ]:
pdf_chunks = chunker.chunk(pdf_doc)

print(f"PDF tree → {len(pdf_chunks)} chunks\n")
for c in pdf_chunks:
    print(f"[{c.index}] type={c.content_type.value:5}  tokens={c.token_count:>3}  │  {c.text[:80]!r}")